# Create an analysis sample from the full dataset

The full file `02_merged_data_full.parquet` is too large to load into memory with `pd.read_parquet` (it raises `ArrowMemoryError`). This notebook builds a smaller **random sample** for analysis without ever holding the whole dataset in memory.

**How it stays within memory:** instead of reading the file at once, we **stream it in batches** with PyArrow. Each row is kept with probability `FRACTION` (a Bernoulli draw), and the kept rows are written straight to the output parquet as we go. At any moment only one batch (`BATCH_SIZE` rows) plus its sampled subset is in RAM — so peak memory is roughly constant regardless of how large the source file is.

The sample is reproducible (fixed `SEED`) and saved to `02_merged_data_sample.parquet`, which the analysis notebooks read instead of the full file.

In [ ]:
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

SOURCE = "../data/02_merged_data_full.parquet"
OUTPUT = "../data/02_merged_data_sample.parquet"

FRACTION = 0.5         # keep ~50% of the rows
SEED = 42              # makes the sample reproducible
BATCH_SIZE = 200_000   # rows read into memory at a time (controls peak memory)

In [ ]:
# Inspect the file via its metadata only — this does NOT read the data into memory.
pf = pq.ParquetFile(SOURCE)
meta = pf.metadata

print(f"{meta.num_rows:,} rows")
print(f"{meta.num_columns} columns")
print(f"{meta.num_row_groups} row groups")
print(f"~{round(meta.num_rows * FRACTION):,} rows expected in the sample (fraction={FRACTION})")

## Stream, sample, and write

We iterate the file batch by batch. For each batch we draw one random number per row and keep the row if it falls below `FRACTION`, then append the kept rows to the output parquet through a single `ParquetWriter`. The full dataset (and the full sample) is never materialised in memory at once.

In [ ]:
rng = np.random.default_rng(SEED)

writer = None
total_rows = 0
kept_rows = 0

try:
    for batch in pf.iter_batches(batch_size=BATCH_SIZE):
        n = batch.num_rows
        total_rows += n

        # Bernoulli keep-mask: each row is kept independently with probability FRACTION.
        mask = pa.array(rng.random(n) < FRACTION)
        sampled = batch.filter(mask)
        kept_rows += sampled.num_rows

        table = pa.Table.from_batches([sampled], schema=pf.schema_arrow)
        if writer is None:
            writer = pq.ParquetWriter(OUTPUT, table.schema)
        writer.write_table(table)
finally:
    if writer is not None:
        writer.close()

print(f"read   {total_rows:,} rows")
print(f"kept   {kept_rows:,} rows ({kept_rows / total_rows:.1%})")
print(f"wrote  -> {OUTPUT}")

In [ ]:
# Verify: the sample is small enough to load normally for analysis.
import pandas as pd

sample = pd.read_parquet(OUTPUT)
print(f"sample shape: {sample.shape[0]:,} rows x {sample.shape[1]} columns")
sample.head()